# CA05 – kNN Based Movie Recommender Engine

Objective: Build a movie recommendation engine using the k-Nearest Neighbors (kNN) algorithm from scikit-learn and identify the 5 movies most similar to The Post based on genre attributes and IMDB rating.

## Step 1 – Import Libraries

We only need two libraries for this task:
- `pandas` — to load the CSV dataset and display results in clean tabular form
- `sklearn.neighbors.NearestNeighbors` — the scikit-learn implementation of the kNN algorithm

No scaling is applied here because genre columns are already binary (0/1) and IMDB ratings are on a bounded numeric scale. For production systems with highly varied feature ranges, normalizing features before running kNN would be important.

In [10]:
# Import the required libraries
import pandas as pd
from sklearn.neighbors import NearestNeighbors

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 2 – Load the Dataset

The dataset is loaded directly from the GitHub URL specified in the assignment. It contains 30 movies, each described by:
- `Movie ID` and `Movie Name` — identifiers only, not used as features
- `IMDB Rating` — continuous numeric feature
- Seven binary genre columns: `Biography`, `Drama`, `Thriller`, `Comedy`, `Crime`, `Mystery`, `History` (1 = Yes, 0 = No)
- `Label` — all zeros; used in classification tasks, ignored here

In [11]:
# Load the movie dataset from the exact URL specified in the assignment
url = "https://github.com/ArinB/MSBA-CA-Data/raw/main/CA05/movies_recommendation_data.csv"
movies = pd.read_csv(url)

print(f"Dataset loaded: {movies.shape[0]} movies x {movies.shape[1]} columns")
print(f"Columns: {movies.columns.tolist()}")
print()

# Display first 5 rows
movies.head()

Dataset loaded: 30 movies x 11 columns
Columns: ['Movie ID', 'Movie Name', 'IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery', 'History', 'Label']



,Movie ID,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
0,58,The Imitation Game,8.0,1,1,1,0,0,0,0,0
1,8,Ex Machina,7.7,0,1,0,0,0,1,0,0
2,46,A Beautiful Mind,8.2,1,1,0,0,0,0,0,0
3,62,Good Will Hunting,8.3,0,1,0,0,0,0,0,0
4,97,Forrest Gump,8.8,0,1,0,0,0,0,0,0


## Step 3 – Explore the Data

Before building the model, we verify that:
1. All feature columns are numeric (kNN requires numeric input)
2. There are no missing values (missing values would corrupt distance calculations)
3. The data structure matches our expectations from the assignment

In [26]:
# Check data types
print("Data types:")
print(movies.dtypes)
print()

# Check for missing values
print("Missing values per column:")
print(movies.isnull().sum())
print()

# Summary statistics for numeric columns
print("Summary statistics:")
print(movies.describe())

Data types:
Movie ID         int64
Movie Name      object
IMDB Rating    float64
Biography        int64
Drama            int64
Thriller         int64
Comedy           int64
Crime            int64
Mystery          int64
History          int64
Label            int64
dtype: object

Missing values per column:
Movie ID       0
Movie Name     0
IMDB Rating    0
Biography      0
Drama          0
Thriller       0
Comedy         0
Crime          0
Mystery        0
History        0
Label          0
dtype: int64

Summary statistics:
        Movie ID  IMDB Rating  Biography      Drama   Thriller     Comedy  \
count  30.000000    30.000000  30.000000  30.000000  30.000000  30.000000   
mean   48.133333     7.696667   0.233333   0.600000   0.100000   0.100000   
std    29.288969     0.666169   0.430183   0.498273   0.305129   0.305129   
min     1.000000     5.900000   0.000000   0.000000   0.000000   0.000000   
25%    27.750000     7.300000   0.000000   0.000000   0.000000   0.000000   
50%    48.

## Step 4 – Prepare the Feature Matrix

We drop the non-feature columns (`Movie ID`, `Movie Name`, `Label`) and keep all eight numerical attributes as the feature matrix X:

| Column | Type | Role |
|---|---|---|
| IMDB Rating | Continuous | Feature |
| Biography | Binary (0/1) | Feature |
| Drama | Binary (0/1) | Feature |
| Thriller | Binary (0/1) | Feature |
| Comedy | Binary (0/1) | Feature |
| Crime | Binary (0/1) | Feature |
| Mystery | Binary (0/1) | Feature |
| History | Binary (0/1) | Feature |

Note: `History` is included because it exists in the actual dataset AND is specified as `Yes` in the query profile for The Post. Omitting it would produce incorrect recommendations.

In [28]:
# Select only the feature columns used for the kNN model
X = movies[["IMDB Rating", "Biography", "Drama", "Thriller",
            "Comedy", "Crime", "Mystery", "History"]]

print("Feature matrix created successfully.")
print(f"Shape of X: {X.shape}")
print(f"Feature columns: {list(X.columns)}")
print()

print(X.head())

Feature matrix created successfully.
Shape of X: (30, 8)
Feature columns: ['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery', 'History']

   IMDB Rating  Biography  Drama  Thriller  Comedy  Crime  Mystery  History
0          8.0          1      1         1       0      0        0        0
1          7.7          0      1         0       0      0        1        0
2          8.2          1      1         0       0      0        0        0
3          8.3          0      1         0       0      0        0        0
4          8.8          0      1         0       0      0        0        0


## Step 5 – Build and Fit the kNN Model

We use scikit-learn's `NearestNeighbors` class, designed for unsupervised nearest-neighbor lookups.

Key parameters:
- `n_neighbors=5` — we want the 5 most similar movies. Since The Post is not in the dataset, we do not need to request an extra neighbor to skip a self-match.
- `metric='euclidean'` — measures straight-line distance in the 8-dimensional feature space.

Euclidean distance formula:  
- For two movies A and B: `distance = sqrt( (a1-b1)^2 + (a2-b2)^2 + ... + (a8-b8)^2 )`
- Smaller distance = more similar movies

In [29]:
# Initialize and fit the kNN model
knn_model = NearestNeighbors(n_neighbors=5, metric="euclidean")
knn_model.fit(X)

print("kNN model fitted successfully.")
print(f"Number of neighbors: {knn_model.n_neighbors}")
print(f"Distance metric: {knn_model.metric}")
print(f"Number of movies used for training: {X.shape[0]}")

kNN model fitted successfully.
Number of neighbors: 5
Distance metric: euclidean
Number of movies used for training: 30


## Step 6 – Define the Query Vector for The Post

We construct the feature vector for The Post using exactly the values given in the assignment. The column order must match the training feature matrix exactly.

Using a DataFrame (rather than a plain array) makes it easy to verify each value is in the correct column, reducing the chance of an ordering mistake.

In [30]:
# Create the feature vector for the query movie "The Post"
the_post = pd.DataFrame([{
    "IMDB Rating": 7.2,
    "Biography": 1,
    "Drama": 1,
    "Thriller": 0,
    "Comedy": 0,
    "Crime": 0,
    "Mystery": 0,
    "History": 1
}])

print("Query movie: The Post")
print(the_post)

Query movie: The Post
   IMDB Rating  Biography  Drama  Thriller  Comedy  Crime  Mystery  History
0          7.2          1      1         0       0      0        0        1


## Step 7 – Find the 5 Most Similar Movies

We call `kneighbors()` on the query vector. This searches the fitted model and returns:
- `distances` — Euclidean distance from The Post to each of the 5 nearest neighbors
- `indices` — row positions (0-indexed) of those neighbors in the original dataset

A smaller distance means the movie is more similar to The Post.

In [31]:
# Make sure the query movie has the same column order as X
the_post = the_post[X.columns]

# Find the 5 nearest neighbors to "The Post"
distances, indices = knn_model.kneighbors(the_post)

print("Nearest neighbors found successfully.")
print(f"Row indices: {indices[0]}")
print(f"Distances: {distances[0].round(4)}")

Nearest neighbors found successfully.
Row indices: [28 27 29 16  2]
Distances: [0.9    1.     1.0198 1.1662 1.4142]


## Step 8 – Display the Recommendations

We use the returned row indices to look up each movie's full details from the original dataset, then attach the computed distance for transparency.

In [32]:
# Get the recommended movies based on the returned indices
recommendations = movies.iloc[indices[0]][
    ["Movie Name", "IMDB Rating", "Biography", "Drama",
     "Thriller", "Comedy", "Crime", "Mystery", "History"]
].copy()

# Add the distance values
recommendations["Distance from The Post"] = distances[0]

# Reset the index and label the ranking
recommendations.reset_index(drop=True, inplace=True)
recommendations.index = recommendations.index + 1
recommendations.index.name = "Rank"

print("Top 5 movie recommendations for 'The Post':")
print(recommendations)

Top 5 movie recommendations for 'The Post':
            Movie Name  IMDB Rating  Biography  Drama  Thriller  Comedy  \
Rank                                                                      
1     12 Years a Slave          8.1          1      1         0       0   
2        Hacksaw Ridge          8.2          1      1         0       0   
3       Queen of Katwe          7.4          1      1         0       0   
4       The Wind Rises          7.8          1      1         0       0   
5     A Beautiful Mind          8.2          1      1         0       0   

      Crime  Mystery  History  Distance from The Post  
Rank                                                   
1         0        0        1                0.900000  
2         0        0        1                1.000000  
3         0        0        0                1.019804  
4         0        0        0                1.166190  
5         0        0        0                1.414214  


## Step 9 – Final Answer

In [33]:
# Print the recommendations in a simple list format
print("Top 5 Movie Recommendations for 'The Post':")
print()

for rank, (_, row) in enumerate(recommendations.iterrows(), start=1):
    print(f"{rank}. {row['Movie Name']} (distance: {row['Distance from The Post']:.4f})")

Top 5 Movie Recommendations for 'The Post':

1. 12 Years a Slave (distance: 0.9000)
2. Hacksaw Ridge (distance: 1.0000)
3. Queen of Katwe (distance: 1.0198)
4. The Wind Rises (distance: 1.1662)
5. A Beautiful Mind (distance: 1.4142)


## Step 10 – Interpretation and Conclusion

### Why these 5 movies?

The kNN algorithm ranks movies based on Euclidean distance in an 8-dimensional feature space. The results are intuitive:

- 12 Years a Slave (distance: 0.90) and Hacksaw Ridge (distance: 1.00) are the closest matches — both are biographical dramas with `History=1`, sharing the most important features with *The Post*. The main difference comes from their higher IMDB ratings (8.1 and 8.2 vs. 7.2).

- Queen of Katwe (1.02) and The Wind Rises (1.17) are also biographical dramas, but they do not include the `History` feature, which increases their distance.

- A Beautiful Mind (1.41) is similarly a biographical drama, but its higher IMDB rating makes it slightly farther in the feature space.

---

### Why is the `History` column important?

The `History` feature plays a key role in the recommendations. Movies that match *The Post* on this feature (such as 12 Years a Slave and Hacksaw Ridge) are ranked highest.

If this column were removed, the recommendations would likely change and become less accurate, showing how even a single feature can significantly influence the results.

---

### Limitations of this approach:

- **Small dataset:** The model is trained on only 30 movies, limiting its generalizability.
- **Binary genre features:** Genres are represented as 0/1, which does not capture the intensity or nuance of a genre.
- **Missing contextual information:** Important factors such as actors, directors, and storyline are not included.
- **No feature scaling:** IMDB ratings (range ~5–9) have more influence than genre variables (0 or 1), which may bias the distance calculation.

---

### At production scale

In real-world systems, platforms like Netflix and Amazon combine collaborative filtering (user behavior), content-based filtering (item similarity), and deep learning techniques to generate more advanced recommendations.

However, kNN demonstrates the core idea clearly: similarity in feature space leads to similar recommendations.

---

### Conclusion

This kNN model effectively demonstrates how similarity-based recommendation systems work. Movies that are closer in feature space are more likely to be recommended together, which aligns well with intuition.